# IST 597: Deep Reinforcement Learning
## Penn State University — Complete Homework Notebook
### Yogeshvar Reddy Kallam

---

This notebook consolidates **all homework assignments** from IST 597 in curriculum order.
Run each section top-to-bottom — dependencies flow naturally downward.

| Section | Topic | Algorithms |
|---------|-------|------------|
| **HW1** | MDPs & Dynamic Programming | Bellman Equations, Policy Evaluation, Policy Improvement |
| **HW2 — Part 1** | REINFORCE with Baseline | Policy Gradient, Variance Reduction |
| **HW2 — Part 2** | Warehouse Robot (Discrete) | PPO via Stable-Baselines3 |
| **HW2 — Part 3** | Partially Observable CartPole | PPO under Partial Observability |
| **HW2 — Part 4** | Warehouse Robot (Mixed Actions) | Actor-Critic, Categorical + Normal Distributions |
| **HW3 — Part 1** | HedgeMaze Multi-Agent | VDN Q-Learning, Cooperative MARL |
| **HW3 — Part 2** | Large FrozenLake | REINFORCE + Optimistic Exploration |
| **HW3 — Part 3** | Offline FrozenLake | Behavioral Cloning, Filtered Cloning |
| **Final** | FrozenLake Delivery | REINFORCE on Multi-Stage Task |
| **Final** | Pendulum Control | Q-Learning with State Discretization |

---


## 🔧 Setup — Install All Dependencies

In [ ]:
# Install all required packages (run once)
import subprocess, sys

packages = ["gymnasium[toy-text]", "stable-baselines3", "torch", "numpy", "matplotlib", "tqdm"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("✅ All packages installed.")


In [ ]:
# ── Global imports used throughout the notebook ──
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical, Normal
from torch.utils.data import Dataset, DataLoader
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Device selection (GPU if available, else CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


---
## 📘 HW1 — MDPs & Dynamic Programming

**Problem:** FrozenLake — 1-row grid with holes, a goal, and a stochastic policy.

**Grid:**
```
H  F  F  S₀  F  F  G
0  1  2   3   4  5  6
```
- `H` = Hole (reward = −1, terminal)
- `F` = Frozen (reward = 0)
- `S₀` = Start (position 3)
- `G` = Goal (reward = +1, terminal)
- Policy: π(left|s) = 0.5, π(right|s) = 0.5 (uniform random)

**Bellman closed-form:**  V^π(S₀) = (Pg − Ph) / (1 − Ps·γ)


In [ ]:
# ── HW1: Analytical Bellman Solution ──────────────────────────────────────

def compute_value(Pg, Ph, gamma):
    """
    Closed-form Bellman solution for the symmetric 1-row FrozenLake.
    V^π(S₀) = (Pg - Ph) / (1 - Ps*γ)
    """
    Ps = 1.0 - Pg - Ph
    if abs(1 - Ps * gamma) < 1e-9:
        return float('inf')
    return (Pg - Ph) / (1 - Ps * gamma)

# Example configuration
Pg    = 1/6   # probability of reaching goal in one transition
Ph    = 1/6   # probability of falling into hole
gamma = 0.9

V = compute_value(Pg, Ph, gamma)
print(f"Pg={Pg:.3f}, Ph={Ph:.3f}, gamma={gamma}")
print(f"V^π(S₀) = {V:.4f}")

# Variance of return G₀
Var_G0 = Pg * (1 - V)**2 + Ph * (-1 - V)**2
print(f"Var(G₀) = {Var_G0:.4f}")

# Minimum episodes needed for |V̂ - V^π| ≤ 0.1 (Central Limit Theorem)
eps = 0.1
N_required = int(np.ceil(Var_G0 / eps**2))
print(f"Min episodes for ε=0.1 accuracy (CLT): N ≈ {N_required:,}")


In [ ]:
# ── HW1: Monte Carlo Policy Evaluation on FrozenLake ─────────────────────

env_hw1 = gym.make("FrozenLake-v1", is_slippery=True)
n_states_hw1 = env_hw1.observation_space.n
n_actions_hw1 = env_hw1.action_space.n
gamma_hw1 = 0.9
n_episodes_hw1 = 50_000

V_mc = np.zeros(n_states_hw1)
V_counts = np.zeros(n_states_hw1)

for _ in range(n_episodes_hw1):
    state, _ = env_hw1.reset()
    trajectory = []
    done = False
    while not done:
        action = env_hw1.action_space.sample()   # uniform random policy
        next_state, reward, done, _, _ = env_hw1.step(action)
        trajectory.append((state, reward))
        state = next_state

    # Every-visit MC update
    G = 0.0
    for s, r in reversed(trajectory):
        G = r + gamma_hw1 * G
        V_mc[s] += G
        V_counts[s] += 1

# Average
with np.errstate(invalid='ignore'):
    V_mc = np.where(V_counts > 0, V_mc / V_counts, 0.0)

print("Monte Carlo Value Estimates (4x4 FrozenLake):")
print(V_mc.reshape(4, 4).round(3))
env_hw1.close()


In [ ]:
# ── HW1: Greedy Policy Improvement ───────────────────────────────────────

env_pi = gym.make("FrozenLake-v1", is_slippery=True)
n_states_pi  = env_pi.observation_space.n
n_actions_pi = env_pi.action_space.n

def policy_evaluation(env, policy, gamma=0.9, theta=1e-4):
    """Iterative policy evaluation — compute V^π."""
    V = np.zeros(env.observation_space.n)
    n_s = env.observation_space.n
    n_a = env.action_space.n
    while True:
        delta = 0
        for s in range(n_s):
            v = 0
            for a in range(n_a):
                prob_a = policy[s, a]
                for prob_trans, s_next, reward, done in env.unwrapped.P[s][a]:
                    v += prob_a * prob_trans * (reward + gamma * (0 if done else V[s_next]))
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < theta:
            break
    return V

def greedy_policy_improvement(env, V, gamma=0.9):
    """One step of greedy policy improvement."""
    n_s = env.observation_space.n
    n_a = env.action_space.n
    policy = np.zeros((n_s, n_a))
    for s in range(n_s):
        Q_s = np.zeros(n_a)
        for a in range(n_a):
            for prob_trans, s_next, reward, done in env.unwrapped.P[s][a]:
                Q_s[a] += prob_trans * (reward + gamma * (0 if done else V[s_next]))
        best_a = np.argmax(Q_s)
        policy[s, best_a] = 1.0
    return policy

# Start with uniform random policy
pi = np.ones((n_states_pi, n_actions_pi)) / n_actions_pi
V_pi = policy_evaluation(env_pi, pi)
pi_improved = greedy_policy_improvement(env_pi, V_pi)

print("Optimal action per state after greedy improvement:")
action_names = ["←", "↓", "→", "↑"]
optimal_actions = np.argmax(pi_improved, axis=1)
print(np.array([action_names[a] for a in optimal_actions]).reshape(4, 4))
env_pi.close()


---
## 📘 HW2 — Part 1: REINFORCE with Baseline (Bandit)

**Problem:** 2-armed bandit — left (reward=2), right (reward=1).  
Softmax policy: π_θ(left) = 1/(1+eθ), π_θ(right) = eθ/(1+eθ)  
At θ=0: both arms equally likely (prob 0.5 each).

**Key result:** Adding baseline b = E[R] = 1.5 reduces gradient variance from **0.5625 → 0**.


In [ ]:
# ── HW2-1: Analytical REINFORCE Gradient Analysis ─────────────────────────

import math

theta = 0.0   # initial parameter
r_left  = 2.0
r_right = 1.0

# Policy probabilities at θ=0
pi_left  = 1 / (1 + math.exp(theta))    # = 0.5
pi_right = math.exp(theta) / (1 + math.exp(theta))  # = 0.5

# Log-policy gradients at θ=0
grad_log_left  = -pi_right   # = -0.5
grad_log_right =  pi_left    # = +0.5

# ── Without baseline ──────────────────────────────────────────────────────
g_left  = r_left  * grad_log_left    # = 2 * (-0.5) = -1.0
g_right = r_right * grad_log_right   # = 1 * (+0.5) = +0.5

E_g       = pi_left * g_left + pi_right * g_right
E_g2      = pi_left * g_left**2 + pi_right * g_right**2
Var_g     = E_g2 - E_g**2

print("── Without Baseline ──────────────────────────────────")
print(f"  Gradient if left  selected: {g_left:.4f}  (prob {pi_left:.2f})")
print(f"  Gradient if right selected: {g_right:.4f}  (prob {pi_right:.2f})")
print(f"  E[∇J]  = {E_g:.4f}")
print(f"  Var[∇J] = {Var_g:.4f}")

# ── With baseline b = E[R] = 1.5 ─────────────────────────────────────────
b = 0.5 * r_left + 0.5 * r_right   # = 1.5
gb_left  = (r_left  - b) * grad_log_left
gb_right = (r_right - b) * grad_log_right

E_gb   = pi_left * gb_left + pi_right * gb_right
E_gb2  = pi_left * gb_left**2 + pi_right * gb_right**2
Var_gb = E_gb2 - E_gb**2

print()
print(f"── With Baseline b = {b} ─────────────────────────────")
print(f"  Gradient if left  selected: {gb_left:.4f}  (prob {pi_left:.2f})")
print(f"  Gradient if right selected: {gb_right:.4f}  (prob {pi_right:.2f})")
print(f"  E[∇J]   = {E_gb:.4f}  ← Same expected gradient!")
print(f"  Var[∇J] = {Var_gb:.4f}  ← Variance collapsed to 0!")

print()
print("✅ Baseline does NOT change the expected gradient,")
print("   but eliminates variance entirely in this case.")


In [ ]:
# ── HW2-1: Live REINFORCE Simulation on 2-Armed Bandit ────────────────────

import torch, torch.optim as optim

def simulate_reinforce(use_baseline=False, n_steps=5000, lr=0.1, seed=42):
    torch.manual_seed(seed)
    theta = torch.tensor([0.0], requires_grad=True)
    optimizer = optim.SGD([theta], lr=lr)
    
    r_vals = {0: 2.0, 1: 1.0}   # left=2, right=1
    b = 1.5  # baseline = E[R]
    
    theta_history = []
    for _ in range(n_steps):
        pi_right = torch.sigmoid(theta)
        pi_left  = 1 - pi_right
        probs = torch.stack([pi_left, pi_right]).squeeze()
        
        action = torch.multinomial(probs, 1).item()
        reward = r_vals[action]
        
        log_prob = torch.log(probs[action])
        advantage = (reward - b) if use_baseline else reward
        loss = -log_prob * advantage
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        theta_history.append(theta.item())
    
    return theta_history

hist_no_base  = simulate_reinforce(use_baseline=False)
hist_baseline = simulate_reinforce(use_baseline=True)

# θ should converge toward +∞ (prefer left arm, reward=2)
plt.figure(figsize=(10, 4))
plt.plot(hist_no_base,  label="No baseline", alpha=0.7)
plt.plot(hist_baseline, label="Baseline b=1.5", alpha=0.7)
plt.xlabel("Step")
plt.ylabel("θ (positive = prefer right, negative = prefer left)")
plt.title("REINFORCE: θ convergence with and without baseline\n(θ→-∞ means policy correctly prefers left arm, reward=2)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Final θ without baseline: {hist_no_base[-1]:.3f}")
print(f"Final θ with baseline:    {hist_baseline[-1]:.3f}")


---
## 📘 HW2 — Part 2: Warehouse Robot with PPO

**Environment:** Custom Gymnasium env — 1D corridor (7 positions).  
Robot picks up packages (random: left or right) from the middle (pos=3) and delivers to pos=0 (left) or pos=6 (right).

**State:** `(position ∈ {0..6}, package_state ∈ {NO_PACKAGE, PACKAGE_LEFT, PACKAGE_RIGHT})`  
**Actions:** MOVE_LEFT, MOVE_RIGHT, PICK_UP, DROP_OFF  
**Algorithm:** PPO via Stable-Baselines3


In [ ]:
# ── HW2-2: Warehouse Robot Environment ───────────────────────────────────

NO_PACKAGE   = 0
PACKAGE_LEFT = 1
PACKAGE_RIGHT = 2

MOVE_LEFT  = 0
MOVE_RIGHT = 1
PICK_UP    = 2
DROP_OFF   = 3

class WarehouseRobotEnv(gym.Env):
    """
    1-D corridor: positions 0..6
    Robot starts at pos=3 (middle), no package.
    PICK_UP at pos=3 assigns a random package (left or right).
    DROP_OFF at correct endpoint gives +1; wrong endpoint gives -0.1.
    5% chance of staying in place on MOVE (stochastic).
    """
    metadata = {'render.modes': ['human']}

    def __init__(self, max_steps=200):
        super().__init__()
        self.observation_space = spaces.MultiDiscrete([7, 3])
        self.action_space      = spaces.Discrete(4)
        self.max_steps = max_steps
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.position     = 3
        self.package      = NO_PACKAGE
        self.current_step = 0
        return (self.position, self.package), {}

    def step(self, action):
        reward = 0

        if action == MOVE_LEFT:
            if random.random() < 0.95:
                self.position = max(self.position - 1, 0)
        elif action == MOVE_RIGHT:
            if random.random() < 0.95:
                self.position = min(self.position + 1, 6)
        elif action == PICK_UP:
            if self.package == NO_PACKAGE and self.position == 3:
                self.package = random.choice([PACKAGE_LEFT, PACKAGE_RIGHT])
        elif action == DROP_OFF:
            if self.package != NO_PACKAGE:
                if ((self.package == PACKAGE_LEFT  and self.position == 0) or
                    (self.package == PACKAGE_RIGHT and self.position == 6)):
                    reward = +1
                else:
                    reward = -0.1
                self.package = NO_PACKAGE

        terminated = False
        self.current_step += 1
        truncated = self.current_step >= self.max_steps
        return (self.position, self.package), reward, terminated, truncated, {}

    def render(self, mode="human"):
        pkg_map = {NO_PACKAGE: "NONE", PACKAGE_LEFT: "LEFT", PACKAGE_RIGHT: "RIGHT"}
        print(f"  pos={self.position}  package={pkg_map[self.package]}")

print("✅ WarehouseRobotEnv defined.")


In [ ]:
# ── HW2-2: Train PPO on Warehouse Robot ──────────────────────────────────

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy as sb3_eval

def make_warehouse_env():
    return WarehouseRobotEnv(max_steps=200)

vec_env_warehouse = DummyVecEnv([make_warehouse_env])

model_warehouse = PPO(
    "MlpPolicy", vec_env_warehouse,
    verbose=0,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
)

print("Training PPO on Warehouse Robot (100,000 timesteps)...")
model_warehouse.learn(total_timesteps=100_000, progress_bar=False)
print("✅ Training complete.")

mean_r, std_r = sb3_eval(model_warehouse, vec_env_warehouse, n_eval_episodes=20, deterministic=True)
print(f"Evaluation (20 episodes): mean reward = {mean_r:.2f} ± {std_r:.2f}")


In [ ]:
# ── HW2-2: Visualise a Warehouse Robot Trajectory ────────────────────────

env_demo = make_warehouse_env()
obs, _ = env_demo.reset()
done = truncated = False
trajectory = []

while not (done or truncated):
    action, _ = model_warehouse.predict(np.array([obs]), deterministic=True)
    next_obs, reward, done, truncated, _ = env_demo.step(int(action[0]))
    trajectory.append((obs, int(action[0]), reward, next_obs))
    obs = next_obs

action_names = {MOVE_LEFT: "MOVE_LEFT", MOVE_RIGHT: "MOVE_RIGHT",
                PICK_UP: "PICK_UP", DROP_OFF: "DROP_OFF"}
pkg_names = {NO_PACKAGE: "NONE", PACKAGE_LEFT: "LEFT", PACKAGE_RIGHT: "RIGHT"}

print(f"{'Step':>4}  {'Pos':>3}  {'Package':>12}  {'Action':>12}  {'Reward':>7}")
print("-" * 52)
for i, (s, a, r, s2) in enumerate(trajectory[:30]):
    print(f"{i:>4}  {s[0]:>3}  {pkg_names[s[1]]:>12}  {action_names[a]:>12}  {r:>7.2f}")
if len(trajectory) > 30:
    print(f"... ({len(trajectory)} total steps)")
print(f"\nTotal reward: {sum(t[2] for t in trajectory):.2f}")


---
## 📘 HW2 — Part 3: Partially Observable CartPole with PPO

**Problem:** Standard CartPole-v1 has 4 observations: `[cart_pos, cart_vel, pole_angle, pole_vel]`.  
We **zero out the velocity signals** (indices 1 and 3), making the task partially observable (POMDP).  
The agent only sees positions, not velocities — can PPO still balance the pole?


In [ ]:
# ── HW2-3: Partial Observation Wrapper ───────────────────────────────────

from gymnasium.wrappers import TransformObservation

def partial_obs(obs: np.ndarray) -> np.ndarray:
    """Zero out cart_vel (index 1) and pole_angular_vel (index 3)."""
    obs = np.array(obs, dtype=np.float32)
    obs[1] = 0.0   # cart velocity  → hidden
    obs[3] = 0.0   # pole ang. vel  → hidden
    return obs

def partial_obs_space() -> spaces.Box:
    low  = np.array([-4.8, 0.0, -0.41887903, 0.0], dtype=np.float32)
    high = np.array([ 4.8, 0.0,  0.41887903, 0.0], dtype=np.float32)
    return spaces.Box(low=low, high=high, shape=(4,), dtype=np.float32)

def make_partial_cartpole():
    env = gym.make("CartPole-v1")
    env = TransformObservation(env, partial_obs, partial_obs_space())
    return env

def make_full_cartpole():
    return gym.make("CartPole-v1")

print("✅ Partial observation CartPole wrapper defined.")
print("   Original obs: [cart_pos, cart_vel, pole_angle, pole_vel]")
print("   Partial  obs: [cart_pos,    0.0  , pole_angle,    0.0  ]")


In [ ]:
# ── HW2-3: Train PPO — Full vs Partial Observation ───────────────────────

# Full observation baseline
vec_full = DummyVecEnv([make_full_cartpole])
model_full = PPO("MlpPolicy", vec_full, verbose=0, gamma=0.99, n_steps=2048,
                 learning_rate=3e-4, batch_size=64, n_epochs=10)
print("Training PPO on FULL observation CartPole (150k steps)...")
model_full.learn(total_timesteps=150_000, progress_bar=False)
mean_full, std_full = sb3_eval(model_full, vec_full, n_eval_episodes=20, deterministic=True)
print(f"  Full obs  → mean reward: {mean_full:.1f} ± {std_full:.1f}")

# Partial observation
vec_partial = DummyVecEnv([make_partial_cartpole])
model_partial = PPO("MlpPolicy", vec_partial, verbose=0, gamma=0.99, n_steps=2048,
                    learning_rate=3e-4, batch_size=64, n_epochs=10)
print("Training PPO on PARTIAL observation CartPole (300k steps)...")
model_partial.learn(total_timesteps=300_000, progress_bar=False)
mean_partial, std_partial = sb3_eval(model_partial, vec_partial, n_eval_episodes=20, deterministic=True)
print(f"  Partial obs → mean reward: {mean_partial:.1f} ± {std_partial:.1f}")

print()
print(f"Performance drop from partial observability: {mean_full - mean_partial:.1f} points")
print("(Max episode reward = 500 steps)")


---
## 📘 HW2 — Part 4: Warehouse Robot with Mixed Action Space (Actor-Critic)

**Challenge:** The action is now a tuple `(action_type: Discrete(3), movement: Continuous[-1,1])`.  
Requires **separate distribution heads** in the policy network:
- `Categorical` distribution for action type (MOVE / PICK_UP / DROP_OFF)
- `Normal` distribution for the continuous movement amount

**Algorithm:** REINFORCE with a Value Function Baseline (Actor-Critic style)


In [ ]:
# ── HW2-4: Mixed-Action Warehouse Environment ─────────────────────────────

class WarehouseRobotMixedEnv(gym.Env):
    """
    State : [position ∈ [0.0, 3.0] (float), package_state ∈ {0,1}]
    Action: (action_type ∈ {MOVE=0, PICK_UP=1, DROP_OFF=2},
             movement   ∈ [-1, 1]  — used only when action_type == MOVE)
    Transitions:
      MOVE     → position += movement + Normal(0, 0.1), clipped to [0,3]
      PICK_UP  → if position ∈ [0.5,1.5] and no package → grab package
      DROP_OFF → if carrying: reward +1 if pos ∈ [2,3], else -0.1
    """
    MOVE=0; PICK_UP=1; DROP_OFF=2
    NO_PACKAGE=0; PACKAGE=1

    def __init__(self, max_steps=200):
        super().__init__()
        self.observation_space = gym.spaces.Tuple((
            gym.spaces.Box(low=0.0, high=3.0, shape=(1,), dtype=np.float32),
            gym.spaces.Discrete(2)
        ))
        self.action_space = gym.spaces.Tuple((
            gym.spaces.Discrete(3),
            gym.spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        ))
        self.max_steps = max_steps
        self.rng = np.random.default_rng()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([1.0, self.NO_PACKAGE], dtype=np.float32)
        self.current_step = 0
        return self.state.copy(), {}

    def step(self, action):
        action_type, movement = action
        reward = 0.0
        position, pkg_state = self.state

        if action_type == self.MOVE:
            noise = self.rng.normal(0, 0.1)
            delta = float(np.clip(movement, -1.0, 1.0)) + noise
            self.state[0] = float(np.clip(position + delta, 0.0, 3.0))
        elif action_type == self.PICK_UP:
            if 0.5 <= position <= 1.5 and pkg_state == self.NO_PACKAGE:
                self.state[1] = self.PACKAGE
        elif action_type == self.DROP_OFF:
            if pkg_state == self.PACKAGE:
                self.state[1] = self.NO_PACKAGE
                reward = 1.0 if 2.0 <= position <= 3.0 else -0.1

        self.current_step += 1
        truncated = self.current_step >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}

print("✅ WarehouseRobotMixedEnv defined.")


In [ ]:
# ── HW2-4: Actor-Critic Policy and Value Networks ────────────────────────

class MixedPolicyNetwork(nn.Module):
    """
    Input  : 2-dim state [position, package_state]
    Outputs: logits (3,) for action_type
             mu (1,) and log_sigma (1,) for movement Normal distribution
    """
    def __init__(self, hidden=256):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.logits_head   = nn.Linear(hidden, 3)
        self.mu_head       = nn.Linear(hidden, 1)
        self.logsig_head   = nn.Linear(hidden, 1)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.logits_head(x), self.mu_head(x), self.logsig_head(x)

class ValueNetwork(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, state):
        return self.net(state)

print("✅ MixedPolicyNetwork and ValueNetwork defined.")


In [ ]:
# ── HW2-4: REINFORCE + Baseline Training Loop ────────────────────────────

def train_mixed_actor_critic(episodes=2000, gamma=0.85, policy_lr=1e-4,
                              vfa_lr=1e-3, print_interval=200):
    env = WarehouseRobotMixedEnv(max_steps=200)
    policy = MixedPolicyNetwork().to(DEVICE)
    vfa    = ValueNetwork().to(DEVICE)
    p_opt  = optim.Adam(policy.parameters(), lr=policy_lr)
    v_opt  = optim.Adam(vfa.parameters(),    lr=vfa_lr)

    reward_history = []
    for ep in range(1, episodes + 1):
        state, _ = env.reset()
        log_probs, rewards, values = [], [], []
        done = truncated = False

        while not (done or truncated):
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE)
            logits, mu, log_sigma = policy(s_t)

            cat  = Categorical(logits=logits)
            a_type = cat.sample()

            sigma = torch.exp(log_sigma.clamp(-4, 2))
            norm  = Normal(mu, sigma)

            if a_type.item() == 0:   # MOVE
                raw_move = norm.sample()
                movement = torch.tanh(raw_move).item()
                log_prob = cat.log_prob(a_type) + norm.log_prob(raw_move).sum()
            else:
                movement = 0.0
                log_prob = cat.log_prob(a_type)

            next_state, r, done, truncated, _ = env.step((a_type.item(), movement))
            log_probs.append(log_prob)
            rewards.append(r)
            values.append(vfa(s_t))
            state = next_state

        # Compute returns and update
        G, p_loss, v_loss = 0.0, 0.0, 0.0
        for t in reversed(range(len(rewards))):
            G = rewards[t] + gamma * G
            adv = G - values[t].detach()
            p_loss += -log_probs[t] * adv
            v_loss += (G - values[t]) ** 2

        p_opt.zero_grad(); p_loss.backward(); p_opt.step()
        v_opt.zero_grad(); v_loss.backward(); v_opt.step()

        reward_history.append(sum(rewards))
        if ep % print_interval == 0:
            avg = np.mean(reward_history[-print_interval:])
            print(f"Episode {ep:>5}/{episodes}  avg reward = {avg:.3f}")

    return policy, vfa, reward_history

print("Starting Actor-Critic training on Mixed-Action Warehouse Robot...")
policy_mixed, vfa_mixed, rewards_mixed = train_mixed_actor_critic(
    episodes=2000, print_interval=200
)
print("✅ Training complete.")

# Plot learning curve
plt.figure(figsize=(10, 4))
window = 100
smoothed = np.convolve(rewards_mixed, np.ones(window)/window, mode='valid')
plt.plot(smoothed, label=f"Smoothed reward (window={window})")
plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("Actor-Critic on Mixed-Action Warehouse Robot")
plt.legend()
plt.tight_layout()
plt.show()


---
## 📘 HW3 — Part 1: Multi-Agent RL — HedgeMaze with VDN

**Task:** Alice and Bob are placed randomly in a 7×7 hedge maze.  
They must learn to meet (occupy the same cell). Reward = +1 when they meet, 0 otherwise.

**Algorithm:** Value Decomposition Network (VDN) Q-Learning  
`Q_tot(s, a_Alice, a_Bob) = Q_Alice(s, a_Alice) + Q_Bob(s, a_Bob)`

Each agent has its own Q-table, trained via a shared team reward signal.  
ε decays from 1.0 → 0.05 over 15,000 episodes to move from exploration to exploitation.


In [ ]:
# ── HW3-1: HedgeMaze Environment ─────────────────────────────────────────

class HedgeMaze:
    """
    7x7 hedge maze. Two agents (Alice, Bob) must navigate to meet.
    Observation: joint integer = alice_loc * SIZE + bob_loc
    Actions: 4 (left=0, right=1, down=2, up=3)
    Reward:  +1 when both agents share the same cell, 0 otherwise.
    """
    HEDGE = [
        "*******",
        "*     *",
        "* *** *",
        "* *   *",
        "*   * *",
        "* *** *",
        "*     *",
        "*******",
    ]
    AGENTS = ["Alice", "Bob"]

    def __init__(self):
        self.HEIGHT = len(self.HEDGE)
        self.WIDTH  = len(self.HEDGE[0])
        self.SIZE   = self.HEIGHT * self.WIDTH
        self.possible_agents = self.AGENTS
        self.agents = []

    def _valid(self, y, x):
        return (0 <= y < self.HEIGHT and 0 <= x < self.WIDTH
                and self.HEDGE[y][x] == " ")

    def _rand_loc(self, exclude=None):
        while True:
            x = random.randrange(self.WIDTH)
            y = random.randrange(self.HEIGHT)
            loc = y * self.WIDTH + x
            if self._valid(y, x) and loc != exclude:
                return loc

    def _obs(self):
        joint = self.locations[0] * self.SIZE + self.locations[1]
        return {a: joint for a in self.AGENTS}

    def action_space(self, agent):
        return spaces.Discrete(4)

    def observation_space(self, agent):
        return spaces.Discrete(self.SIZE * self.SIZE)

    def reset(self):
        self.agents = list(self.AGENTS)
        self.locations = [self._rand_loc()]
        self.locations.append(self._rand_loc(exclude=self.locations[0]))
        return self._obs(), {}

    def _move(self, loc, action):
        x, y = loc % self.WIDTH, loc // self.WIDTH
        dx, dy = [(-1,0),(1,0),(0,1),(0,-1)][action]
        nx, ny = x + dx, y + dy
        if self._valid(ny, nx):
            return ny * self.WIDTH + nx
        return loc

    def step(self, actions):
        for i, agent in enumerate(self.AGENTS):
            self.locations[i] = self._move(self.locations[i], actions[agent])

        done = self.locations[0] == self.locations[1]
        reward = 1.0 if done else 0.0
        rewards      = {a: reward for a in self.AGENTS}
        terminations = {a: done   for a in self.AGENTS}
        if done:
            self.agents = []
        return self._obs(), rewards, terminations, {a: False for a in self.AGENTS}, {}

    def render(self):
        grid = [list(row) for row in self.HEDGE]
        icons = ["A", "B"]
        for i, loc in enumerate(self.locations):
            y, x = loc // self.WIDTH, loc % self.WIDTH
            grid[y][x] = icons[i]
        for row in grid:
            print("".join(row))

print("✅ HedgeMaze environment defined.")

# Quick sanity check with random actions
env_hm = HedgeMaze()
obs, _ = env_hm.reset()
print(f"Initial joint obs: Alice={obs['Alice']}, Bob={obs['Bob']}")
for _ in range(3):
    acts = {a: env_hm.action_space(a).sample() for a in env_hm.agents}
    obs, rews, terms, _, _ = env_hm.step(acts)
print("Random steps OK.")


In [ ]:
# ── HW3-1: VDN Q-Tables ──────────────────────────────────────────────────

class QTable(nn.Module):
    """Learnable Q-table stored as nn.Parameter for gradient-based updates."""
    def __init__(self, n_states, n_actions):
        super().__init__()
        self.Q = nn.Parameter(torch.zeros(n_states, n_actions))

    def forward(self, state):
        return self.Q[state]

# Initialise one Q-table per agent
env_hm_init = HedgeMaze()
agents_hm = env_hm_init.possible_agents
hm_models = {}
for agent in agents_hm:
    n_s = env_hm_init.observation_space(agent).n
    n_a = env_hm_init.action_space(agent).n
    hm_models[agent] = QTable(n_s, n_a).to(DEVICE)

all_params = [p for m in hm_models.values() for p in m.parameters()]
hm_optimizer = optim.Adam(all_params, lr=0.0005)
hm_loss_fn   = nn.MSELoss()

print(f"Q-table sizes:")
for agent in agents_hm:
    print(f"  {agent}: {list(hm_models[agent].Q.shape)}")


In [ ]:
# ── HW3-1: VDN Q-Learning Training ───────────────────────────────────────

HM_GAMMA       = 0.9
HM_EPISODES    = 25_000
HM_EPS_START   = 1.0
HM_EPS_END     = 0.05
HM_EPS_DECAY   = 15_000
PRINT_INTERVAL = 1000

env_hm_train = HedgeMaze()
episode_lengths = []

print(f"Training VDN on HedgeMaze ({HM_EPISODES:,} episodes)...")

for episode in range(HM_EPISODES):
    obs, _ = env_hm_train.reset()
    obs_t  = {a: torch.tensor(obs[a], dtype=torch.long, device=DEVICE)
               for a in agents_hm}

    epsilon = max(HM_EPS_END,
                  HM_EPS_START - (HM_EPS_START - HM_EPS_END) * (episode / HM_EPS_DECAY))
    terminated = False
    length = 0

    while not terminated:
        length += 1
        actions = {}
        q_current = []

        for agent in env_hm_train.agents:
            if random.random() < epsilon:
                a = env_hm_train.action_space(agent).sample()
            else:
                with torch.no_grad():
                    a = torch.argmax(hm_models[agent](obs_t[agent])).item()
            actions[agent] = a
            q_current.append(hm_models[agent](obs_t[agent])[a])

        next_obs, rewards, terminations, _, _ = env_hm_train.step(actions)
        reward     = rewards[agents_hm[0]]   # shared reward
        terminated = any(terminations.values())

        next_obs_t = {a: torch.tensor(next_obs[a], dtype=torch.long, device=DEVICE)
                      for a in agents_hm}

        # VDN target: shared reward + γ * Σ max Q_i(s')
        with torch.no_grad():
            if terminated:
                q_target = torch.tensor(reward, dtype=torch.float32, device=DEVICE)
            else:
                q_next_sum = sum(
                    torch.max(hm_models[a](next_obs_t[a])) for a in agents_hm
                )
                q_target = reward + HM_GAMMA * q_next_sum

        q_sum_current = torch.stack(q_current).sum()
        loss = hm_loss_fn(q_sum_current, q_target)

        hm_optimizer.zero_grad()
        loss.backward()
        hm_optimizer.step()

        obs_t = next_obs_t

    episode_lengths.append(length)
    if (episode + 1) % PRINT_INTERVAL == 0:
        avg = np.mean(episode_lengths[-PRINT_INTERVAL:])
        print(f"  Episode {episode+1:>6} | ε={epsilon:.3f} | avg steps to meet: {avg:.1f}")

print("\n✅ VDN training complete.")


In [ ]:
# ── HW3-1: Test Learned Policy ────────────────────────────────────────────

env_hm_test = HedgeMaze()
obs, _ = env_hm_test.reset()
obs_t  = {a: torch.tensor(obs[a], dtype=torch.long, device=DEVICE)
           for a in agents_hm}

print("Greedy policy test — Alice and Bob navigating maze:\n")
step = 0
MAX_STEPS = 50

while env_hm_test.agents and step < MAX_STEPS:
    actions = {}
    for agent in env_hm_test.agents:
        with torch.no_grad():
            a = torch.argmax(hm_models[agent](obs_t[agent])).item()
        actions[agent] = a

    next_obs, rewards, terminations, _, _ = env_hm_test.step(actions)
    obs_t = {a: torch.tensor(next_obs[a], dtype=torch.long, device=DEVICE)
              for a in agents_hm}
    step += 1

env_hm_test.render()
print(f"\nMet after {step} steps  (reward={rewards.get('Alice', 0):.1f})")

# Learning curve
plt.figure(figsize=(10, 4))
window = 500
smoothed_hm = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')
plt.plot(smoothed_hm)
plt.xlabel("Episode")
plt.ylabel("Steps to meet")
plt.title("HedgeMaze VDN: Steps to meet Alice & Bob")
plt.tight_layout()
plt.show()


---
## 📘 HW3 — Part 2: Large FrozenLake with Optimistic Exploration

**Problem:** 10×10 FrozenLake with custom hole placement.  
Standard REINFORCE struggles because rewards are extremely sparse — a random agent almost never reaches the goal.

**Solution — Optimistic Exploration Bonus:**  
`augmented_reward = real_reward + β / √(visit_count(state))`  

- Rare (unvisited) states get a high bonus → agent is "optimistic" about their value
- Bonus shrinks as states are visited → exploration fades naturally
- β = 0.1 (selected hyperparameter)


In [ ]:
# ── HW3-2: 10×10 FrozenLake Environment Setup ────────────────────────────

MAP_10x10 = [
    'SHFFFFFHFF',
    'FFFFFFFFHH',
    'FFFHHHFFFF',
    'FFFFFFFFHF',
    'FFFFFFHFHF',
    'FFFFFFFFFH',
    'FFFFFHHFHF',
    'FHFFFFFFFF',
    'FFFHFFFFFF',
    'FFFFHFFFHG',
]

env_large = gym.make('FrozenLake-v1', desc=MAP_10x10, is_slippery=True)
n_states_lg  = env_large.observation_space.n   # 100
n_actions_lg = env_large.action_space.n        # 4

print(f"Large FrozenLake: {n_states_lg} states, {n_actions_lg} actions")
print("Map (S=start, H=hole, F=frozen, G=goal):")
for row in MAP_10x10:
    print(" ", row)


In [ ]:
# ── HW3-2: Policy Network + State Visit Counter ───────────────────────────

class LargeLakePolicyNet(nn.Module):
    def __init__(self, n_states, n_actions):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(n_states, n_actions))

    def forward(self, state):
        return F.softmax(self.logits[state], dim=-1)

# Hyperparameters
LG_GAMMA          = 0.99
LG_LR             = 0.01
LG_EPISODES       = 25_000
LG_BETA           = 0.1    # exploration bonus weight
LG_PRINT_INTERVAL = 2000

policy_lg  = LargeLakePolicyNet(n_states_lg, n_actions_lg).to(DEVICE)
optimizer_lg = optim.Adam(policy_lg.parameters(), lr=LG_LR)

# Visit counts — initialised to 1 to avoid division-by-zero
state_counts = np.ones(n_states_lg)

print("✅ Large FrozenLake policy and visit counters initialised.")
print(f"   β = {LG_BETA}  (exploration bonus = β / √count)")


In [ ]:
# ── HW3-2: REINFORCE with Optimistic Exploration ─────────────────────────

lg_actual_returns  = []   # real env rewards (no bonus)
lg_interval_actual = []

print(f"Training REINFORCE + Optimistic Exploration on 10×10 FrozenLake "
      f"({LG_EPISODES:,} episodes)...")

for episode in range(LG_EPISODES):
    state, _ = env_large.reset()
    log_probs      = []
    rewards_bonus  = []    # used for training (includes bonus)
    actual_rewards = []    # real rewards for reporting

    state_counts[state] += 1
    done = False

    while not done:
        probs_np = policy_lg(state).detach().cpu().numpy()
        probs_np = np.clip(probs_np, 1e-8, None)
        probs_np /= probs_np.sum()
        action = np.random.choice(n_actions_lg, p=probs_np)

        next_state, reward, done, _, _ = env_large.step(action)

        # Exploration bonus
        bonus = LG_BETA / np.sqrt(state_counts[state])
        rewards_bonus.append(reward + bonus)
        actual_rewards.append(reward)

        log_prob = torch.log(policy_lg(state)[action])
        log_probs.append(log_prob)

        state = next_state
        state_counts[state] += 1

    # REINFORCE update on augmented rewards
    G, loss = 0.0, 0.0
    for t in range(len(rewards_bonus) - 1, -1, -1):
        G = rewards_bonus[t] + LG_GAMMA * G
        loss += -log_probs[t] * G

    optimizer_lg.zero_grad()
    loss.backward()
    optimizer_lg.step()

    ep_actual = sum(actual_rewards)
    lg_actual_returns.append(ep_actual)
    lg_interval_actual.append(ep_actual)

    if (episode + 1) % LG_PRINT_INTERVAL == 0:
        avg = np.mean(lg_interval_actual)
        print(f"  Episode {episode+1:>6} | avg actual return = {avg:.5f} "
              f"| visits coverage: {(state_counts > 2).sum()}/{n_states_lg} states")
        lg_interval_actual = []

env_large.close()
print("\n✅ Training complete.")


In [ ]:
# ── HW3-2: Results Visualisation ─────────────────────────────────────────

window = 1000
smoothed_lg = np.convolve(lg_actual_returns, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(smoothed_lg)
plt.xlabel("Episode")
plt.ylabel("Actual Episode Return (no bonus)")
plt.title(f"Large FrozenLake — REINFORCE + Optimistic Exploration (β={LG_BETA})")
plt.tight_layout()
plt.show()

print(f"Final {window}-episode average return: {np.mean(lg_actual_returns[-window:]):.5f}")
print(f"States visited at least twice: {(state_counts > 2).sum()} / {n_states_lg}")


---
## 📘 HW3 — Part 3: Offline FrozenLake — Behavioral Cloning & Filtered Cloning

**Setting:** Offline RL — no environment interaction during training.  
Given a fixed dataset of trajectories from a (weak) behavior policy π_β.

**Three approaches:**
1. **Estimate** the expected return of π_β from the data
2. **Behavioral Cloning (BC):** Supervised learning on all (state, action) pairs
3. **Filtered Cloning (FBC):** Only train on *successful* trajectories (return > 0)

Filtering removes corrupting signal from failed trajectories and learns only from what worked.

> 📝 **Note:** If `trajectories.pkl` is not found, a synthetic dataset is generated to demonstrate the full pipeline.


In [ ]:
# ── HW3-3: Load or Generate Trajectories ─────────────────────────────────

import pickle, os

TRAJ_FILE = 'trajectories.pkl'

def generate_synthetic_trajectories(n=100_000, seed=42):
    """
    Generate synthetic FrozenLake trajectories from a near-random behavior policy.
    Expected return ≈ 0.02 (mirrors the homework hint of ~0.0226).
    """
    rng = random.Random(seed)
    env = gym.make("FrozenLake-v1", is_slippery=True)
    trajs = []
    for _ in range(n):
        state, _ = env.reset()
        traj = []
        done = False
        while not done:
            # Near-random policy with slight bias toward right and down
            weights = [0.2, 0.3, 0.3, 0.2]   # LEFT, DOWN, RIGHT, UP
            action = rng.choices(range(4), weights=weights)[0]
            next_state, reward, done, _, _ = env.step(action)
            traj.append((state, action, reward))
            state = next_state
            if len(traj) > 100:   # cap length
                break
        trajs.append(traj)
    env.close()
    return trajs

if os.path.exists(TRAJ_FILE):
    with open(TRAJ_FILE, 'rb') as f:
        trajectories = pickle.load(f)
    print(f"✅ Loaded {len(trajectories):,} trajectories from {TRAJ_FILE}")
else:
    print(f"⚠️  {TRAJ_FILE} not found — generating synthetic trajectories...")
    trajectories = generate_synthetic_trajectories(n=100_000)
    print(f"✅ Generated {len(trajectories):,} synthetic trajectories.")

# Part (a): Estimate expected return of behavior policy
total_ret = sum(sum(step[2] for step in traj) for traj in trajectories)
est_return_beta = total_ret / len(trajectories)
print(f"\n--- Part (a) ---")
print(f"Estimated expected return of π_β: {est_return_beta:.4f}")
print(f"(Hint: expected ~0.0226 for the course dataset)")


In [ ]:
# ── HW3-3: Environment + Policy Network for Offline RL ───────────────────

env_offline = gym.make("FrozenLake-v1", is_slippery=True)
N_S_OFF = env_offline.observation_space.n   # 16
N_A_OFF = env_offline.action_space.n        # 4

class OfflinePolicyNetwork(nn.Module):
    """Tabular policy stored as logits — one row per state."""
    def __init__(self, n_states, n_actions):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(n_states, n_actions))

    def forward(self, state):
        if isinstance(state, (int, np.integer)):
            return F.softmax(self.logits[state], dim=-1)
        return F.softmax(self.logits[state], dim=-1)

class TrajDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        s, a = self.data[idx]
        return torch.tensor(s, dtype=torch.long), torch.tensor(a, dtype=torch.long)

def evaluate_offline_policy(policy, n_episodes=20_000):
    """Monte-Carlo evaluation of a learned policy."""
    policy.eval()
    returns = []
    for _ in range(n_episodes):
        state, _ = env_offline.reset()
        total, done = 0.0, False
        while not done:
            with torch.no_grad():
                probs = policy(state).cpu().numpy()
                probs = np.clip(probs, 1e-8, None)
                probs /= probs.sum()
            action = np.random.choice(N_A_OFF, p=probs)
            state, reward, done, _, _ = env_offline.step(action)
            total += reward
        returns.append(total)
    policy.train()
    return float(np.mean(returns))

print("✅ Offline RL setup complete.")
print(f"   FrozenLake: {N_S_OFF} states, {N_A_OFF} actions")


In [ ]:
# ── HW3-3: Part (b) — Behavioral Cloning ─────────────────────────────────

print("--- Part (b): Behavioral Cloning ---")

# Build (state, action) dataset from ALL trajectories
bc_data = [(s, a) for traj in trajectories for (s, a, _) in traj]
bc_dataset    = TrajDataset(bc_data)
bc_dataloader = DataLoader(bc_dataset, batch_size=1024, shuffle=True, num_workers=0)

bc_policy    = OfflinePolicyNetwork(N_S_OFF, N_A_OFF).to(DEVICE)
bc_optimizer = optim.Adam(bc_policy.parameters(), lr=1e-2)
bc_criterion = nn.CrossEntropyLoss()

print(f"Training on {len(bc_data):,} (state, action) pairs...")
for epoch in range(10):
    total_loss = 0
    for states, actions in bc_dataloader:
        states, actions = states.to(DEVICE), actions.to(DEVICE)
        logits = bc_policy.logits[states]
        loss   = bc_criterion(logits, actions)
        bc_optimizer.zero_grad(); loss.backward(); bc_optimizer.step()
        total_loss += loss.item()
    print(f"  BC Epoch {epoch+1:2}/10 | loss = {total_loss/len(bc_dataloader):.4f}")

avg_return_bc = evaluate_offline_policy(bc_policy, n_episodes=20_000)
print(f"\nBehavioral Cloning average return: {avg_return_bc:.4f}")
print(f"Behavior policy estimated return:  {est_return_beta:.4f}")


In [ ]:
# ── HW3-3: Part (c) — Filtered Behavioral Cloning ────────────────────────

print("--- Part (c): Filtered Behavioral Cloning ---")

# Keep only SUCCESSFUL trajectories (return > 0)
successful = [t for t in trajectories if sum(s[2] for s in t) > 0]
print(f"Successful trajectories: {len(successful):,} / {len(trajectories):,} "
      f"({100*len(successful)/len(trajectories):.2f}%)")

if len(successful) == 0:
    print("⚠️  No successful trajectories — using fallback BC policy.")
    fbc_policy = OfflinePolicyNetwork(N_S_OFF, N_A_OFF).to(DEVICE)
else:
    fbc_data       = [(s, a) for traj in successful for (s, a, _) in traj]
    fbc_dataset    = TrajDataset(fbc_data)
    fbc_dataloader = DataLoader(fbc_dataset, batch_size=min(512, len(fbc_dataset)),
                                shuffle=True, num_workers=0)

    fbc_policy    = OfflinePolicyNetwork(N_S_OFF, N_A_OFF).to(DEVICE)
    fbc_optimizer = optim.Adam(fbc_policy.parameters(), lr=1e-2)
    fbc_criterion = nn.CrossEntropyLoss()

    print(f"Training Filtered BC on {len(fbc_data):,} (state, action) pairs...")
    for epoch in range(15):
        total_loss = 0
        for states, actions in fbc_dataloader:
            states, actions = states.to(DEVICE), actions.to(DEVICE)
            logits = fbc_policy.logits[states]
            loss   = fbc_criterion(logits, actions)
            fbc_optimizer.zero_grad(); loss.backward(); fbc_optimizer.step()
            total_loss += loss.item()
        print(f"  FBC Epoch {epoch+1:2}/15 | loss = {total_loss/len(fbc_dataloader):.4f}")

    avg_return_fbc = evaluate_offline_policy(fbc_policy, n_episodes=20_000)
    print(f"\n--- Offline RL Summary ---")
    print(f"  π_β  (behavior)         return: {est_return_beta:.4f}")
    print(f"  π_BC (cloned)           return: {avg_return_bc:.4f}")
    print(f"  π*   (filtered cloning) return: {avg_return_fbc:.4f}  ← target ≥ 0.035")

env_offline.close()


---
## 📘 Final Project — Part 1: FrozenLakeDelivery with REINFORCE

**Custom multi-stage task** on a 5×5 grid.

```
G   T     G = Goal (deliver here)   T = Tree (wall)
  HH      H = Hole (-1, terminal)   K = Key
 H        C = Chest (pick up gift)
  HH
K   C
```

**Progress stages:** INITIAL → (reach K) → HAS_KEY → (reach C) → HAS_PRESENT → (reach G) → DELIVERED  
**Reward shaping:** +0.2 (key), +0.3 (chest), +1.0 (goal) — helps guide REINFORCE through the sparse landscape.  
**Algorithm:** REINFORCE with a 3D policy table `logits[position][progress][action]`


In [ ]:
# ── Final-1: FrozenLakeDelivery Environment ──────────────────────────────

INITIAL          = 0
HAS_KEY          = 1
HAS_PRESENT      = 2
DELIVERED_PRESENT = 3

ACTION_TO_DELTA  = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}
PERPENDICULAR    = {0:[3,1], 1:[0,2], 2:[3,1], 3:[0,2]}

class FrozenLakeDeliveryEnv(gym.Env):
    metadata = {"render.modes": ["human"]}

    def __init__(self):
        super().__init__()
        self.desc = ["G   T", "  HH ", " H   ", "  HH ", "K   C"]
        self.n_rows = len(self.desc)
        self.n_cols = len(self.desc[0])
        self.nS     = self.n_rows * self.n_cols   # 25
        self.observation_space = spaces.Tuple((
            spaces.Discrete(self.nS),
            spaces.Discrete(4)
        ))
        self.action_space = spaces.Discrete(4)
        self.reset()

    def reset(self, seed=None, options=None):
        self.rng   = random.Random(seed)
        self.state = (0, INITIAL)
        self.done  = False
        return self.state, {}

    def step(self, action):
        if self.done:
            raise RuntimeError("Call reset() first.")
        directions = [action] + PERPENDICULAR[action]
        actual_movement = self.rng.choice(directions)

        position, progress = self.state
        x, y = position % self.n_cols, position // self.n_cols
        dx, dy = ACTION_TO_DELTA[actual_movement]
        nx, ny = x + dx, y + dy
        if not (0 <= nx < self.n_cols and 0 <= ny < self.n_rows):
            nx, ny = x, y

        cell = self.desc[ny][nx]
        reward = 0

        if   cell == "G" and progress == DELIVERED_PRESENT: reward = 1.0
        elif cell == "K" and progress == INITIAL:           reward = 0.2
        elif cell == "C" and progress == HAS_KEY:           reward = 0.3
        elif cell == "H":                                    reward = -1.0

        if cell == "H":
            self.done = True
        elif cell == "K" and progress == INITIAL:           progress = HAS_KEY
        elif cell == "C" and progress == HAS_KEY:           progress = HAS_PRESENT
        elif cell == "G" and progress == DELIVERED_PRESENT: self.done = True

        self.state = (nx + ny * self.n_cols, progress)
        return self.state, reward, self.done, False, {}

    def render(self):
        position, progress = self.state
        grid = [list(row) for row in self.desc]
        grid[position // self.n_cols][position % self.n_cols] = "P"
        print("\n".join("".join(row) for row in grid))
        progress_labels = {0:"INITIAL", 1:"HAS_KEY", 2:"HAS_PRESENT", 3:"DELIVERED"}
        print(f"Progress: {progress_labels[progress]}")

print("✅ FrozenLakeDeliveryEnv defined.")
env_fl = FrozenLakeDeliveryEnv()
state, _ = env_fl.reset()
env_fl.render()


In [ ]:
# ── Final-1: REINFORCE Training on FrozenLakeDelivery ────────────────────

FL_GAMMA    = 0.99
FL_LR       = 0.01
FL_EPISODES = 25_000
FL_PRINT    = 2000

env_fl = FrozenLakeDeliveryEnv()
state_shape = (env_fl.observation_space[0].n,   # 25  (positions)
               env_fl.observation_space[1].n)    # 4   (progress stages)
n_actions_fl = env_fl.action_space.n             # 4

class DeliveryPolicyNet(nn.Module):
    """Logit table indexed by (position, progress) → 4 action logits."""
    def __init__(self):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(state_shape + (n_actions_fl,)))

    def forward(self, state):
        pos, prog = state
        return F.softmax(self.logits[pos, prog], dim=-1)

policy_fl   = DeliveryPolicyNet().to(DEVICE)
optimizer_fl = optim.Adam(policy_fl.parameters(), lr=FL_LR)

fl_episode_rewards = []
fl_interval_rewards = []

print(f"Training REINFORCE on FrozenLakeDelivery ({FL_EPISODES:,} episodes)...")

for episode in range(FL_EPISODES):
    state, _ = env_fl.reset()
    log_probs, rewards = [], []
    done = False

    while not done:
        probs  = policy_fl(state)
        action = np.random.choice(n_actions_fl, p=probs.detach().cpu().numpy())
        next_state, reward, done, _, _ = env_fl.step(action)
        log_probs.append(torch.log(probs[action]))
        rewards.append(reward)
        state = next_state

    # REINFORCE loss
    G, loss = 0.0, 0.0
    for t in range(len(rewards) - 1, -1, -1):
        G = rewards[t] + FL_GAMMA * G
        loss += -log_probs[t] * G

    optimizer_fl.zero_grad()
    loss.backward()
    optimizer_fl.step()

    ep_ret = sum(rewards)
    fl_episode_rewards.append(ep_ret)
    fl_interval_rewards.append(ep_ret)

    if (episode + 1) % FL_PRINT == 0:
        avg = np.mean(fl_interval_rewards)
        print(f"  Episode {episode+1:>6} | avg reward = {avg:.5f}")
        fl_interval_rewards = []

print("\n✅ Training complete.")

# Learning curve
window = 1000
smoothed_fl = np.convolve(fl_episode_rewards, np.ones(window)/window, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(smoothed_fl)
plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("FrozenLake Delivery — REINFORCE")
plt.tight_layout()
plt.show()


---
## 📘 Final Project — Part 2: Pendulum with Discretized Q-Learning

**Challenge:** Pendulum-v1 has **continuous** state AND action — incompatible with Q-tables.  
**Solution:** Discretize both into a finite grid.

- **State bins:** 10 × 10 × 10 over `[cos(θ), sin(θ), θ̇]` → 1,000 discrete states
- **Action bins:** 5 discrete torque values `{−2, −1, 0, 1, 2}` Nm

Then apply standard **Q-Learning** with ε-greedy exploration.


In [ ]:
# ── Final-2: Pendulum Q-Learning with State Discretization ───────────────

NUM_ACTIONS_P  = 5
NUM_BINS       = [10, 10, 10]   # cos(θ), sin(θ), θ̇
ACTION_SPACE_P = np.linspace(-2, 2, NUM_ACTIONS_P)

env_pendulum = gym.make("Pendulum-v1")
Q_pendulum   = np.zeros(NUM_BINS + [NUM_ACTIONS_P])

# Hyperparameters
P_ALPHA    = 0.1
P_GAMMA    = 0.99
P_EPSILON  = 0.2
P_EPISODES = 5_000
P_PRINT    = 1000

def discretize_pendulum(state):
    """Map continuous pendulum state to discrete bin indices."""
    cos_theta, sin_theta, theta_dot = state
    bin_edges = [
        np.linspace(-1, 1, NUM_BINS[0] + 1)[1:-1],
        np.linspace(-1, 1, NUM_BINS[1] + 1)[1:-1],
        np.linspace(-8, 8, NUM_BINS[2] + 1)[1:-1],
    ]
    return tuple(np.digitize(s, b) for s, b in zip(state, bin_edges))

total_rewards_p = []

print(f"Training Q-Learning on Pendulum-v1 ({P_EPISODES:,} episodes)...")

for episode in range(P_EPISODES):
    state, _ = env_pendulum.reset()
    state_d   = discretize_pendulum(state)
    total_r   = 0.0

    for t in range(200):
        if np.random.rand() < P_EPSILON:
            action_idx = np.random.choice(NUM_ACTIONS_P)
        else:
            action_idx = np.argmax(Q_pendulum[state_d])

        action    = np.array([ACTION_SPACE_P[action_idx]])
        next_state, reward, done, _, _ = env_pendulum.step(action)
        next_d    = discretize_pendulum(next_state)

        best_next = np.argmax(Q_pendulum[next_d])
        Q_pendulum[state_d][action_idx] += P_ALPHA * (
            reward + P_GAMMA * Q_pendulum[next_d][best_next]
            - Q_pendulum[state_d][action_idx]
        )

        state_d = next_d
        total_r += reward
        if done:
            break

    total_rewards_p.append(total_r)
    if (episode + 1) % P_PRINT == 0:
        avg = np.mean(total_rewards_p[-P_PRINT:])
        print(f"  Episode {episode+1:>6} | avg reward = {avg:.1f}")

env_pendulum.close()
print("\n✅ Q-Learning complete.")

# Plot
window = 500
smoothed_p = np.convolve(total_rewards_p, np.ones(window)/window, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(smoothed_p)
plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("Pendulum — Discretized Q-Learning")
plt.axhline(-200, color='r', linestyle='--', label='Random baseline (~-1200)')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Final {window}-episode avg reward: {np.mean(total_rewards_p[-window:]):.1f}")
print("(Perfect upright pendulum = 0; random = ~-1200)")


---
## 📊 Course Summary

| Section | Algorithm | Key Concept |
|---------|-----------|-------------|
| HW1 | Bellman Equations, MC Evaluation, Policy Improvement | MDP foundations, closed-form value functions |
| HW2-1 | REINFORCE with Baseline | Baseline reduces variance without biasing gradient |
| HW2-2 | PPO (Stable-Baselines3) | Clipped surrogate objective, stable policy updates |
| HW2-3 | PPO under POMDP | Partial observability degrades performance |
| HW2-4 | Actor-Critic, Mixed Actions | Categorical + Normal distributions, advantage baseline |
| HW3-1 | VDN Q-Learning | Q_tot = Σ Q_i, cooperative MARL with implicit coordination |
| HW3-2 | REINFORCE + Exploration | Optimistic bonus β/√count drives sparse reward exploration |
| HW3-3 | Behavioral / Filtered Cloning | Offline RL: filtering bad data dramatically improves policy |
| Final-1 | REINFORCE, reward shaping | Multi-stage tasks, shaped rewards guide learning |
| Final-2 | Q-Learning + Discretization | Continuous → discrete state/action for tabular RL |

---
*IST 597: Deep Reinforcement Learning — Penn State University, Spring 2025*
